## Cross-Machine Peer-to-Peer Communication

Same workflow as `peer_active_policy.ipynb`, but the two policies live
on **separate physical (or virtual) machines** connected over a real
network.  Because the connection is full-duplex, both nodes are equal
peers — either can call into the other.

### Setup

| Node | What to run | Where |
|------|-------------|-------|
| **Node 1** | Run **Part A** cells — starts communication first. | e.g. `192.168.1.10` |
| **Node 2** | Run **Part B** cells — initiates the peering handshake. | e.g. `192.168.1.20` |

Before running, edit the placeholder IPs in the configuration cell
below.  Node 1's port is chosen automatically; copy the printed port
and secret into Node 2's cell.

In [ ]:
import uuid
import time
import laila
from laila.macros.defaults import DefaultPolicy, DefaultPool, DefaultTCPIPProtocol

---
### Configuration — edit these

Replace the placeholder IPs with the actual addresses of your
two machines.  Both machines must be able to reach each other on
the chosen port (check firewalls / security-groups).

In [ ]:
NODE1_IP = "192.168.1.10"   # <-- machine running Part A
NODE2_IP = "192.168.1.20"   # <-- machine running Part B

---
## Part A — Run on Node 1

Creates the policy, starts communication, memorizes a local entry,
and prints the **port** and **secret** that Node 2 needs to peer.

In [ ]:
node1 = DefaultPolicy()
node1_tcp = DefaultTCPIPProtocol(
    host=NODE1_IP,
    port=0,
    peer_secret_key=uuid.uuid4().hex,
)

laila.active_policy = node1
laila.communication.add_connection(node1_tcp)

node1_pool = DefaultPool()
laila.memory.add_pool(pool=node1_pool, affinity=1.0, pool_nickname="node1-store")

entry = laila.constant(data={"message": "stored on Node 1"}, nickname="cross-machine-entry")
with laila.guarantee:
    laila.memorize(entries=entry, pool_nickname="node1-store")

print("Node 1 policy:", node1.global_id)
print("Entry        :", entry.global_id)
print("Pool has it  :", entry.global_id in node1_pool.resource)

In [ ]:
print("========================================")
print("  Node 1 is ready.")
print(f"  PORT  : {node1_tcp.port}")
print(f"  SECRET: {node1_tcp.peer_secret_key}")
print("========================================")
print("Copy the PORT and SECRET into Node 2's notebook.")

---
## Part B — Run on Node 2

Creates the policy, peers with Node 1 using the printed port and
secret, switches the active policy to the remote proxy, and queries
Node 1's memory over the network.

In [ ]:
# Paste the PORT and SECRET from Node 1's output
NODE1_PORT   = 0       # <-- replace with Node 1's printed PORT
NODE1_SECRET = ""      # <-- replace with Node 1's printed SECRET

In [ ]:
node2 = DefaultPolicy()
node2_tcp = DefaultTCPIPProtocol(
    host=NODE2_IP,
    port=0,
    peer_secret_key=NODE1_SECRET,
)

laila.active_policy = node2
laila.communication.add_connection(node2_tcp)

print("Node 2 policy:", node2.global_id)

In [ ]:
# Peer with Node 1 over the real network
remote_node1_id = laila.communication.add_tcpip_peer(NODE1_IP, NODE1_PORT, NODE1_SECRET)
time.sleep(0.5)

print(f"Peered with Node 1: {remote_node1_id}")
print(f"Node 2 peers: {list(laila.peers)}")

In [ ]:
# Switch the active policy to the remote Node 1 proxy
remote_node1 = laila.peers[remote_node1_id]
laila.active_policy = remote_node1

print("Active policy:", laila.active_policy.global_id, "(Node 1 via proxy)")

In [ ]:
# Query Node 1's pool router remotely — the entry should be there
routed_pool = remote_node1.central.memory.pool_router.route(
    entries=["any"], pool_nickname="node1-store"
)

print("Remote pool data:")
for k, v in routed_pool.items():
    if k == "resource":
        print(f"  resource: {len(v)} entries stored")
    else:
        print(f"  {k}: {v}")

### Bidirectional — Node 1 can also reach Node 2

Back on **Node 1**, once the peering is established, Node 1 can call
into Node 2 the same way.  Run the cell below on Node 1's notebook:

In [ ]:
# --- Run this cell on Node 1 after Node 2 has peered ---

print("Node 1 peers:", list(node1.central.communication.peers))

node2_id = list(node1.central.communication.peers)[0]
remote_node2 = node1.central.communication.peers[node2_id]

print(f"Node 1 sees peer: {remote_node2.global_id}")

---
### Cleanup

Run the appropriate cell on each machine.

In [ ]:
# Node 2 cleanup
laila.active_policy = node2
laila.communication.remove_connection(node2_tcp)
print("Node 2 disconnected.")

In [ ]:
# Node 1 cleanup
laila.active_policy = node1
laila.communication.remove_connection(node1_tcp)
print("Node 1 stopped.")